# PHASE 2 — LLM-Based Aspect & Sentiment Labeling

### Overview
1. Load the cleaned dataset (output of Phase 1) and cap extreme-length reviews before sending to the LLM.
2. Define the 7-aspect taxonomy, prompt template, and JSON parsing/extraction helpers.
3. Validate the extraction pipeline: single review → 20-review smoke test → 100-review batch → 1,000-review random batch.
4. Supplement the random batch with **keyword-targeted** batches for aspects that are naturally rare
   (packaging, irritation, breakouts, scent), since a purely random sample under-represents them.
5. Combine and export the labeled dataset.
6. **Iteration 1**: after Phase 3 evaluation showed `irritation`'s negative class was too thin to learn well,
   run a second targeted batch specifically for negative-irritation language, merge, and re-export (`_v2`).
7. **Iteration 2**: same for `packaging`'s negative class, producing the final labeled dataset (`_v3`),
   which is what Phase 3 trains on.

### Step 1: Load dataset

In [1]:
# Imports needed for data processing and LLM calls
import pandas as pd
from tqdm import tqdm
import os
import ollama
import json
import re


In [2]:
# Load cleaned reviews CSV into a DataFrame
df = pd.read_csv("skincare_reviews_clean.csv")

### Step 2: Check review length, and cap extreme outliers before sending to the LLM

In [3]:
# Inspect review length percentiles to find long-tail outliers
print(df["review_text"].str.split().str.len().quantile([0.90, 0.95, 0.99, 1.0]))

0.90    112.0
0.95    141.0
0.99    221.0
1.00    982.0
Name: review_text, dtype: float64


In [4]:
# Cap only the long-tail outliers before sending to the LLM, to bound runtime without
# truncating the vast majority of reviews. 300 words covers well above the 99th
# percentile (~221 words); only the most extreme outliers are affected.

def truncate_review(text, max_words=300):
    words = text.split()
    if len(words) <= max_words:
        return text
    return " ".join(words[:max_words]) + "..."

df["review_text_for_llm"] = df["review_text"].apply(truncate_review)

### Step 3: Aspect taxonomy, prompt template, and JSON parsing helpers

In [5]:
# Build LLM prompt template and helper functions for JSON extraction

ASPECTS = ["texture", "scent", "breakouts", "irritation", "efficacy", "packaging", "value"]

PROMPT_TEMPLATE = """You are an expert at analyzing skincare product reviews.

Given a review, identify which of the following aspects are explicitly mentioned, using these definitions:

- texture: how the product physically feels or applies (e.g. greasy, lightweight, sticky, absorbs fast, leaves residue)
- scent: smell or fragrance of the product, whether present, strong, pleasant, or absent
- breakouts: specifically acne, pimples, or clogged pores caused or helped by the product
- irritation: adverse skin reactions OTHER than acne — redness, stinging, burning, itching, sensitivity
- efficacy: whether the product delivered visible results (e.g. brightening, hydration, anti-aging, evening skin tone)
- packaging: the container or dispenser itself (e.g. pump vs. jar, leaking, hygiene, travel-friendliness)
- value: whether the product was worth the price (cost-to-quality judgment), not just a mention of price

For each aspect that IS mentioned, classify the sentiment as "positive" or "negative" based only on
what the review says about that specific aspect, not the review overall.

Sentiment direction for breakouts and irritation: "positive" means the product did NOT cause
breakouts/irritation, or improved existing breakouts/irritation. "negative" means it caused or
worsened breakouts/irritation.

Do not include aspects that are not mentioned. Do not guess or infer aspects that aren't explicitly
discussed in the text.

Respond ONLY with valid JSON in this exact format, nothing else before or after:
{{
  "texture": "positive" | "negative" | null,
  "scent": "positive" | "negative" | null,
  "breakouts": "positive" | "negative" | null,
  "irritation": "positive" | "negative" | null,
  "efficacy": "positive" | "negative" | null,
  "packaging": "positive" | "negative" | null,
  "value": "positive" | "negative" | null
}}

Review: "{review_text}"
"""

def extract_json(text):
    """Try to pull a JSON object out of the model's response, even if it added extra text."""
    match = re.search(r"\{.*\}", text, re.DOTALL)
    if not match:
        return None
    try:
        return json.loads(match.group())
    except json.JSONDecodeError:
        return None

def normalize_null(value):
    if isinstance(value, str) and value.strip().lower() in {"null", "none", "n/a", ""}:
        return None
    return value

def extract_aspects(review_text, model="llama3.1:8b"):
    prompt = PROMPT_TEMPLATE.format(review_text=review_text)
    response = ollama.chat(
        model=model,
        messages=[{"role": "user", "content": prompt}],
        options={"temperature": 0}
    )
    raw = response["message"]["content"]
    parsed = extract_json(raw)

    if parsed is None:
        return {aspect: "PARSE_ERROR" for aspect in ASPECTS}

    return {aspect: normalize_null(parsed.get(aspect)) for aspect in ASPECTS}

### Step 4: Smoke test on a single review

In [6]:
# Quick smoke-test on a sample review
sample = "Love the smell of this but it broke me out within a week. Packaging is nice though, feels expensive."
result = extract_aspects(sample)
print(result)

{'texture': None, 'scent': 'positive', 'breakouts': 'negative', 'irritation': None, 'efficacy': None, 'packaging': 'positive', 'value': None}


### Step 5: Batch extraction with checkpointing (resumable)

In [ ]:
# Batch extraction with checkpointing (can resume interrupted runs)

def run_batch_extraction(
    df,
    text_col="review_text_for_llm",
    checkpoint_path="llm_labels_checkpoint.csv",
    model="llama3.1:8b",
    checkpoint_every=25
):
    """
    Runs aspect extraction on all reviews and saves progress
    so the process can continue after an interruption.
    """

    # Check if a previous checkpoint already exists
    if os.path.exists(checkpoint_path):
        results_df = pd.read_csv(checkpoint_path, index_col=0)  # Load previous results
        done_indices = set(results_df.index)  # Get already processed review IDs
        print(f"Resuming: {len(done_indices)} reviews already labeled.")
    else:
        results_df = pd.DataFrame(columns=ASPECTS)  # Start empty results table
        done_indices = set()

    # Keep only reviews that have not been processed yet
    remaining = df[~df.index.isin(done_indices)]
    print(f"{len(remaining)} reviews left to process.")

    results_buffer = []  # Temporary storage before saving

    # Loop through each remaining review
    for idx, row in tqdm(remaining.iterrows(), total=len(remaining)):

        try:
            # Send the review to the LLM
            labels = extract_aspects(row[text_col], model=model)

        except Exception as e:
            # If something fails, mark the review as ERROR
            labels = {aspect: "ERROR" for aspect in ASPECTS}
            print(f"Error on index {idx}: {e}")

        labels["review_index"] = idx  # Keep the original review index
        results_buffer.append(labels)  # Add result to the buffer

        # Save progress after every 25 reviews
        if len(results_buffer) >= checkpoint_every:
            batch_df = pd.DataFrame(results_buffer).set_index("review_index")
            results_df = pd.concat([results_df, batch_df])  # Add new labels
            results_df.to_csv(checkpoint_path)  # Save checkpoint to disk
            results_buffer = []  # Clear the buffer

    # Save any remaining reviews that didn't reach 25
    if results_buffer:
        batch_df = pd.DataFrame(results_buffer).set_index("review_index")
        results_df = pd.concat([results_df, batch_df])
        results_df.to_csv(checkpoint_path)

    # Return the complete labeled dataset
    return results_df

### Step 6: Small validation runs before committing to a full batch
20 reviews → spot-check → 100 reviews → check error rate & aspect frequency

In [8]:
# Run extraction on a small sample and return top results
sample_20 = df.sample(20, random_state=42)
results = run_batch_extraction(sample_20, checkpoint_path="llm_labels_test.csv")
results.head(20)

Resuming: 20 reviews already labeled.
0 reviews left to process.


0it [00:00, ?it/s]


,texture,scent,breakouts,irritation,efficacy,packaging,value
344257,positive,NaN,NaN,NaN,positive,NaN,positive
30231,positive,NaN,NaN,NaN,NaN,NaN,NaN
31385,NaN,NaN,NaN,positive,positive,NaN,NaN
383468,NaN,NaN,negative,NaN,negative,NaN,NaN
41206,NaN,NaN,NaN,NaN,positive,NaN,NaN
156268,positive,positive,NaN,NaN,positive,NaN,positive
198612,negative,NaN,NaN,NaN,NaN,NaN,negative
131329,positive,positive,NaN,NaN,positive,positive,positive
356536,positive,negative,NaN,NaN,positive,NaN,positive
871982,NaN,NaN,NaN,NaN,NaN,NaN,negative


In [9]:
# Inspect example reviews and their extracted labels (ensure indices exist in `results`)
for idx in [344257, 156268, 489527, 41206]:
    print("REVIEW:", df.loc[idx, "review_text"][:300])
    print("LABELS:", results.loc[idx].to_dict())
    print("---")

REVIEW: I have combo skin and this felt incredible on. It was smooth, non sticky and very hydrating as well. I love that I can use it everyday. Fresh products have been a part of my skincare routine and this one will definitely be included from here on. I loved that it fit my skin type and that it helps red
LABELS: {'texture': 'positive', 'scent': nan, 'breakouts': nan, 'irritation': nan, 'efficacy': 'positive', 'packaging': nan, 'value': 'positive'}
---
REVIEW: If you know anyone addicted to that horrible chapstick, then buy them this. it is matte, smells wonderful (not too girly), and conditions very well, plus it has sunblock. I was impressed enough that I’ll try their other products. Fair price too.
LABELS: {'texture': 'positive', 'scent': 'positive', 'breakouts': nan, 'irritation': nan, 'efficacy': 'positive', 'packaging': nan, 'value': 'positive'}
---
REVIEW: Love the bottle it’s really pretty, the scent is nice not overwhelming. My skin feels softer and I can see overall improve

In [10]:
# Run extraction on the 100-review sample
sample_100 = df.sample(100, random_state=42)

results_100 = run_batch_extraction(
    sample_100,
    text_col="review_text_for_llm",
    checkpoint_path="llm_labels_sample100.csv"
)

Resuming: 100 reviews already labeled.
100 reviews left to process.


100%|██████████| 100/100 [07:16<00:00,  4.37s/it]


In [11]:
# 1. Any parse errors or runtime errors?
for aspect in ASPECTS:
    print(aspect, (results_100[aspect] == "PARSE_ERROR").sum(), "parse errors,",
                  (results_100[aspect] == "ERROR").sum(), "errors")

# 2. Aspect frequency — how often is each aspect actually mentioned?
for aspect in ASPECTS:
    print(f"\n{aspect}:")
    print(results_100[aspect].value_counts(dropna=False))

texture 0 parse errors, 0 errors
scent 0 parse errors, 0 errors
breakouts 0 parse errors, 0 errors
irritation 0 parse errors, 0 errors
efficacy 0 parse errors, 0 errors
packaging 0 parse errors, 0 errors
value 0 parse errors, 0 errors

texture:
texture
positive    103
NaN          78
negative     19
Name: count, dtype: int64

scent:
scent
NaN         148
positive     37
negative     15
Name: count, dtype: int64

breakouts:
breakouts
NaN         155
positive     31
negative     14
Name: count, dtype: int64

irritation:
irritation
NaN         171
positive     26
negative      3
Name: count, dtype: int64

efficacy:
efficacy
positive    145
NaN          30
negative     25
Name: count, dtype: int64

packaging:
packaging
NaN         175
positive     20
negative      5
Name: count, dtype: int64

value:
value
NaN         108
positive     67
negative     25
Name: count, dtype: int64


### Step 7: Random 1,000-review batch

In [12]:
# Run extraction on the full 1,000-review sample
sample_1000 = df.sample(1000, random_state=42)

results_1000 = run_batch_extraction(
    sample_1000,
    text_col="review_text_for_llm",
    checkpoint_path="llm_labels_sample1000.csv"
)

Resuming: 1000 reviews already labeled.
0 reviews left to process.


0it [00:00, ?it/s]


In [13]:
# Any parse errors or runtime errors?
for aspect in ASPECTS:
    print(aspect, (results_1000[aspect] == "PARSE_ERROR").sum(), "parse errors,",
                  (results_1000[aspect] == "ERROR").sum(), "errors")

# Aspect frequency — how often is each aspect actually mentioned?
for aspect in ASPECTS:
    print(f"\n{aspect}:")
    print(results_1000[aspect].value_counts(dropna=False))

texture 0 parse errors, 0 errors
scent 0 parse errors, 0 errors
breakouts 0 parse errors, 0 errors
irritation 0 parse errors, 0 errors
efficacy 0 parse errors, 0 errors
packaging 0 parse errors, 0 errors
value 0 parse errors, 0 errors

texture:
texture
positive    489
NaN         410
negative    101
Name: count, dtype: int64

scent:
scent
NaN         751
positive    189
negative     60
Name: count, dtype: int64

breakouts:
breakouts
NaN         818
positive    122
negative     60
Name: count, dtype: int64

irritation:
irritation
NaN         866
positive     99
negative     35
Name: count, dtype: int64

efficacy:
efficacy
positive    716
NaN         187
negative     97
Name: count, dtype: int64

packaging:
packaging
NaN         888
positive     84
negative     28
Name: count, dtype: int64

value:
value
NaN         603
positive    291
negative    106
Name: count, dtype: int64


In [14]:
# Normalize null-like strings in `results_1000` and print counts
for aspect in ASPECTS:
    results_1000[aspect] = results_1000[aspect].apply(normalize_null)

# Confirm it's fixed
for aspect in ASPECTS:
    print(aspect, results_1000[aspect].value_counts(dropna=False))

texture texture
positive    489
NaN         410
negative    101
Name: count, dtype: int64
scent scent
NaN         751
positive    189
negative     60
Name: count, dtype: int64
breakouts breakouts
NaN         818
positive    122
negative     60
Name: count, dtype: int64
irritation irritation
NaN         866
positive     99
negative     35
Name: count, dtype: int64
efficacy efficacy
positive    716
NaN         187
negative     97
Name: count, dtype: int64
packaging packaging
NaN         888
positive     84
negative     28
Name: count, dtype: int64
value value
NaN         603
positive    291
negative    106
Name: count, dtype: int64


### Step 8: Targeted supplementary batch for sparse aspects
A random sample under-represents aspects people don't always mention (packaging, irritation,
breakouts, scent). Pre-filter candidates by keyword, then let the LLM make the actual
mention/sentiment judgment — keywords only bias *which* reviews get sampled, not the label.

In [15]:
# Keyword sets for each sparse aspect (used only to find candidate reviews to prioritize —
# the LLM still does the actual aspect/sentiment judgment, keywords just bias sampling)
aspect_keywords = {
    "packaging": ["pump", "bottle", "jar", "packaging", "dispenser", "leak", "tube", "cap", "container"],
    "irritation": ["sting", "burn", "itchy", "irritat", "red", "sensitive", "rash", "reaction"],
    "breakouts": ["breakout", "broke me out", "acne", "pimple", "clogged pore", "zit"],
    "scent": ["smell", "scent", "fragrance", "fragrant", "odor"],
}

# Reviews already labeled — exclude these from the new sample
already_labeled_idx = set(sample_1000.index)

candidates = {}
for aspect, keywords in aspect_keywords.items():
    mask = df["review_text"].str.contains("|".join(keywords), case=False, na=False)
    pool = df[mask & ~df.index.isin(already_labeled_idx)]
    candidates[aspect] = pool
    print(f"{aspect}: {len(pool)} candidate reviews found")

packaging: 124469 candidate reviews found
irritation: 293895 candidate reviews found
breakouts: 132476 candidate reviews found
scent: 240595 candidate reviews found


In [16]:
sample_sizes = {"packaging": 150, "irritation": 150, "breakouts": 100, "scent": 100}

supplementary_parts = []
for aspect, n in sample_sizes.items():
    pool = candidates[aspect]
    take = min(n, len(pool))
    part = pool.sample(take, random_state=42)
    supplementary_parts.append(part)

supplementary_sample = pd.concat(supplementary_parts).drop_duplicates(subset="review_text")
print("Total supplementary batch size:", len(supplementary_sample))

Total supplementary batch size: 500


In [17]:
results_supp = run_batch_extraction(
    supplementary_sample,
    text_col="review_text_for_llm",
    checkpoint_path="llm_labels_supplementary.csv"
)

for aspect in ASPECTS:
    results_supp[aspect] = results_supp[aspect].apply(normalize_null)

for aspect in ASPECTS:
    print(f"\n{aspect}:")
    print(results_supp[aspect].value_counts(dropna=False))

Resuming: 500 reviews already labeled.
0 reviews left to process.


0it [00:00, ?it/s]


texture:
texture
positive    243
NaN         206
negative     51
Name: count, dtype: int64

scent:
scent
NaN         291
positive    168
negative     41
Name: count, dtype: int64

breakouts:
breakouts
NaN         323
positive    135
negative     42
Name: count, dtype: int64

irritation:
irritation
NaN         369
positive     95
negative     36
Name: count, dtype: int64

efficacy:
efficacy
positive    365
NaN          94
negative     41
Name: count, dtype: int64

packaging:
packaging
NaN         362
positive     92
negative     46
Name: count, dtype: int64

value:
value
NaN         277
positive    164
negative     58
neutral       1
Name: count, dtype: int64


Drop any rows with schema-invalid values (e.g. a stray `"neutral"`) before merging.

In [18]:
valid = {"positive", "negative"}
invalid_any = pd.Series(False, index=results_supp.index)

for aspect in ASPECTS:
    invalid_mask = results_supp[aspect].notna() & ~results_supp[aspect].isin(valid)

    if invalid_mask.any():
        print(aspect, "unexpected values:",
              results_supp.loc[invalid_mask, aspect].unique())

    invalid_any |= invalid_mask

results_supp = results_supp.loc[~invalid_any]

value unexpected values: <ArrowStringArray>
['neutral']
Length: 1, dtype: str


In [19]:
# Merge supplementary samples with their labels and save to CSV
supp_final = supplementary_sample.merge(
    results_supp,
    left_index=True,
    right_index=True
)

supp_final.to_csv("skincare_reviews_llm_supplementary.csv", index=False)

### Step 9: Merge and export the combined labeled dataset

In [20]:
# Combine main and supplementary labeled sets and export final CSV
main = sample_1000.merge(results_1000, left_index=True, right_index=True)
supp = supplementary_sample.merge(results_supp, left_index=True, right_index=True)

final_labeled = (
    pd.concat([main, supp])
      .drop_duplicates(subset="review_text")
)

final_labeled.to_csv("skincare_reviews_llm_final.csv", index=False)

### Step 10 — Iteration 1: targeted labeling for `irritation` (negative class)

Phase 3's first training run (on the dataset saved above) showed `irritation`'s negative class
was far too sparse to learn — only ~71 negative examples in the whole dataset, giving test-set
precision 0.17 / recall 0.08 / F1 0.11 for that class. Rather than a broad keyword net (which
mostly catches *positive* irritation mentions, e.g. "didn't irritate my sensitive skin"), this
batch uses phrases that are specifically negative-reaction language.

In [21]:
# Load your existing full labeled dataset (from the LLM labeling notebook)
df_labeled = pd.read_csv("skincare_reviews_llm_final.csv")
already_labeled_idx = set(df_labeled.index)  # adjust if index isn't preserved — see note below

# Reload the full cleaned dataset to search for new candidates
df_full = pd.read_csv("skincare_reviews_clean.csv")

strong_negative_irritation_keywords = [
    "broke out in a rash", "allergic reaction", "burned my skin", "burning sensation",
    "made my skin red and", "had to stop using", "caused irritation", "irritated my skin",
    "stung so bad", "skin reacted badly", "caused a reaction", "made my face burn",
    "left my skin red and irritated", "caused breakouts and irritation",
    "my skin got worse", "flared up", "had a bad reaction"
]

mask = df_full["review_text"].str.contains("|".join(strong_negative_irritation_keywords), case=False, na=False)
strong_negative_candidates = df_full[mask & ~df_full.index.isin(already_labeled_idx)]
print("New candidates found:", len(strong_negative_candidates))

New candidates found: 5968


In [22]:
already_labeled_texts = set(df_labeled["review_text"])
mask = df_full["review_text"].str.contains("|".join(strong_negative_irritation_keywords), case=False, na=False)
strong_negative_candidates = df_full[mask & ~df_full["review_text"].isin(already_labeled_texts)]
print("New candidates found:", len(strong_negative_candidates))

New candidates found: 5959


In [23]:
N = 300
sample_size = min(N, len(strong_negative_candidates))
irritation_supp_sample = strong_negative_candidates.sample(sample_size, random_state=99)

irritation_supp_sample["review_text_for_llm"] = irritation_supp_sample["review_text"].apply(truncate_review)

results_irritation_supp = run_batch_extraction(
    irritation_supp_sample,
    text_col="review_text_for_llm",
    checkpoint_path="llm_labels_irritation_supp.csv"
)

for aspect in ASPECTS:
    results_irritation_supp[aspect] = results_irritation_supp[aspect].apply(normalize_null)

# Check whether the targeting actually worked
print(results_irritation_supp["irritation"].value_counts(dropna=False))

Resuming: 300 reviews already labeled.
0 reviews left to process.


0it [00:00, ?it/s]

irritation
negative    140
positive    122
NaN          38
Name: count, dtype: int64


In [24]:
print(results_irritation_supp["irritation"].value_counts(dropna=False))

irritation
negative    140
positive    122
NaN          38
Name: count, dtype: int64


Result: 140 negative / 122 positive / 38 not-mentioned out of 300 — a much higher negative concentration than the broad keyword pass. Merge and export:

In [25]:
irritation_supp_labeled = irritation_supp_sample.merge(
    results_irritation_supp, left_index=True, right_index=True
)

df_labeled_updated = pd.concat([df_labeled, irritation_supp_labeled]).drop_duplicates(subset="review_text")
print("Before:", df_labeled.shape, "After:", df_labeled_updated.shape)

df_labeled_updated.to_csv("skincare_reviews_llm_final_v2.csv", index=False)

Before: (1499, 23) After: (1799, 23)


### Step 11 — Iteration 2: targeted labeling for `packaging` (negative class)

Retraining on the `_v2` dataset fixed `irritation` (F1 0.11 → 0.82) but `packaging`'s negative
class slipped slightly (this batch wasn't targeted at packaging at all, and packaging's negative
class was already thin — only ~73 examples). Same playbook, this time with packaging
failure-mode language (leaked, pump broke, arrived damaged, etc.).

In [26]:
df_labeled = pd.read_csv("skincare_reviews_llm_final_v2.csv")  # your current 1,799-row file
already_labeled_texts = set(df_labeled["review_text"])

df_full = pd.read_csv("skincare_reviews_clean.csv")

strong_negative_packaging_keywords = [
    "pump broke", "pump doesn't work", "pump stopped working", "pump is broken",
    "leaked", "leaking", "leaks everywhere", "arrived broken", "arrived damaged",
    "cracked bottle", "cap doesn't", "hard to get the product out", "hard to dispense",
    "doesn't pump", "flimsy packaging", "packaging is cheap", "bottle broke",
    "jar cracked", "spilled", "wasted product because of the", "difficult to open",
    "impossible to get the last", "can't get the product out"
]

mask = df_full["review_text"].str.contains("|".join(strong_negative_packaging_keywords), case=False, na=False)
packaging_candidates = df_full[mask & ~df_full["review_text"].isin(already_labeled_texts)]
print("New candidates found:", len(packaging_candidates))

New candidates found: 1518


In [27]:
N = 300
sample_size = min(N, len(packaging_candidates))
packaging_supp_sample = packaging_candidates.sample(sample_size, random_state=17)
packaging_supp_sample["review_text_for_llm"] = packaging_supp_sample["review_text"].apply(truncate_review)

results_packaging_supp = run_batch_extraction(
    packaging_supp_sample,
    text_col="review_text_for_llm",
    checkpoint_path="llm_labels_packaging_supp.csv"
)

for aspect in ASPECTS:
    results_packaging_supp[aspect] = results_packaging_supp[aspect].apply(normalize_null)

print(results_packaging_supp["packaging"].value_counts(dropna=False))

Resuming: 300 reviews already labeled.
0 reviews left to process.


0it [00:00, ?it/s]

packaging
negative    257
positive     40
NaN           3
Name: count, dtype: int64


Result: 257 negative / 40 positive / 3 not-mentioned out of 300 — very sharp targeting. Merge and export the final dataset used for training:

In [28]:
packaging_supp_labeled = packaging_supp_sample.merge(
    results_packaging_supp, left_index=True, right_index=True
)

df_labeled_updated_v3 = pd.concat([df_labeled, packaging_supp_labeled]).drop_duplicates(subset="review_text")
print("Before:", df_labeled.shape, "After:", df_labeled_updated_v3.shape)

df_labeled_updated_v3.to_csv("skincare_reviews_llm_final_v3.csv", index=False)

Before: (1799, 23) After: (2099, 23)
